<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="40%"></a>
</p>

# Duckietown and ROS

In this notebook we will use what we have learned about ROS in the previous notebooks to start to control the Duckiebot, either in the Duckiematrix or on real hardware. 

## See what the Duckiebot Sees

Run the following command to build your code (we will write the code below but we can already build and run it):

    dts code build -R ROBOTNAME

The first execution will take some time, as it is building a Docker image, but we only need to do this once. 

Following that we can run the code with:

    dts code workbench -R ROBOTNAME [-m]

Where `ROBOTNAME` can be either a virtual or real Duckiebot name. If you are using a virtual robot running the Duckiematrix, you should include the `-m` flag (the Duckiematrix needs to be already running: `dts code start_matrix`). This command synchronizes code to your Duckiebot and so it will ask you for the `ssh` password, which is `quackquack` if you used the default value. 

You can run the noVNC desktop (again in a new terminal) with

    dts code vnc -R ROBOTNAME

Once you open the noVNC browser you should see the same icons as for previous notebooks in this LX on the left hand side. Double-click on the one that says `RQT Image ...`, clicking on this opens the [rqt_image_view](https://wiki.ros.org/rqt_image_view) ROS utility.

In the top left scroll down bar of the `rqt_image_view` window you should see only one option: `/agent/camera_node/image/compressed` if you select it you will see the output from the camera on the Duckiebot.

![rqt_image_view](../assets/rqt_image_view_duckiematrix.png)

## Terminals Recap

You should have *3* terminals open at this point (*4* if you are running in the Duckiematrix):

1. The VSCode editor (`dts code editor`)
2. The running code (`dts code workbench`)
3. The noVNC server (`dts code vnc`)
4. (optional) The Duckiematrix (`dts code start_matrix`)

## Try out the Joystick

You can also open the joystick by double clicking on the `Joystick` icon. You can try clicking on the directions or using the arrow keys on your keyboard. You should see the corresponding direction on the joystick turn green, but you won't see the robot moving in the simulation (assuming you still have the `rqt_image_view` window open from the previous step).

However, the joystick **is** doing something. To see what it is doing, open up a terminal by clicking on the `LXTerminal` icon as before. Execute the following command on the terminal:

    rostopic echo /ROBOTNAME/joy

Then click back to the joystick window and start clicking on the directions and look closely at the output that is produced in the terminal. In particular, look at the `axes` part of the data that is produced. You should see one of the values changing (the position may change and the value may be either a `+1.0` or a `-1.0` depending on which direction you push). 

![joy_echo](../assets/joy_echo.png)

Take a careful note of which location in the `axes` data changes to which value when push each of the four directions on the joystick. We will need that information later. 

Finally, do the following in the terminal

    rostopic list

to see all of the topics. It will be a long list but the two that are concerned with here are

```
/ROBOTNAME/joy [sensor_msgs/Joy]
/ROBOTNAME/wheels_driver_node/wheels_cmd [duckietown_msgs/WheelsCmdStamped]
```

We can see the topic `/ROBOTNAME/joy` and we know at least how that data is being published. This message is of type [Joy.msg](https://docs.ros.org/en/noetic/api/sensor_msgs/html/msg/Joy.html) (this is a standard datatype provided by ROS). The topic that we need to publish messages on to make the robot actually move is `/ROBOTNAME/wheels_driver_node/wheels_cmd`. This message is of type [WheelsCmdStamped.msg](https://github.com/duckietown/dt-ros-commons/blob/daffy/packages/duckietown_msgs/msg/WheelsCmdStamped.msg) (this is a message that we defined and is available to us because it is defined in an upstream Docker image).

### The Task Ahead

Our objective for the rest of this exercise will be to create a ROS node to take the messages that are being published on the `/ROBOTNAME/joy` topic and use them to publish something to the `/ROBOTNAME/wheels_driver_node/wheels_cmd` topic so that the robot moves and we can control it with the joystick.  

## Writing our ROS Node

Use the EXPLORER menu on the left of this notebook to find the file `packages` -> `src/dt-joystick-demo` -> `src` -> `dt-joystick-demo-node.py`. Click on the `joystick-demo-node.py`  file to open it (you may want to split screen using the button at the top right ). For now it's empty. 

We need to start with some basics. First off we need the following at the top of the file:

```python
#!/usr/bin/env python3
```
This tells the Python interpreter that this is a Python file. 

Next we will need to import some packages, these will be `rospy`, `os`, and then the types of *messages* we will be using later.

```python
import rospy
import os
from sensor_msgs.msg import Joy
from duckietown_msgs.msg import WheelsCmdStamped
```

At the bottom of your file let's build the main function which will be called when your Python script is executed. It should look something like this:

```python
if __name__ == "__main__":
    # Initialize the node
    node = DTJoystickDemoNode()
    rospy.init_node('dt-joystick-demo-node')
    # Keep it spinning
    rospy.spin()
```

This code initializes a class of type `DTJoystickDemoNode` (which we have yet to define), and then does some ROS-specific stuff, namely initializes the node with `rospy.init_node` which registers the node with a unique name, and then finally starts the node running with the `rospy.spin()` function. This will ensure that we get data from topics we subscribe to and that data that we publish actually gets sent (among other things).

Now we are ready to define the class called `DTJoystickDemoNode`. We can start with this code:

```python
class DTJoystickDemoNode():
    def __init__(self):
```

### Building the publisher and subscriber

The main thing we need to do in the `__init__` function is setup the publisher and subscriber. To do that we are going to need to know the name of our robot. This is stored as an environment variable, which we can get with:

```python
        veh_name = os.environ["VEHICLE_NAME"]
```
 

Now let's build the subscriber. It should look something like this:

```python
        self.sub_joy = rospy.Subscriber(
            "f/{veh_name}/joy", 
            Joy,
            self.process_joy
        )
```

**Note**: As with all Python code make sure to get the indentation right

This initializes a `Subscriber`, tells it to listen to the topic `{veh_name}/joy`, tells it that the data coming on the topic should be of type `Joy`, and that we will process it in a function `self.process_joy` (this is called a function callback). 

The publisher should look like this:

```python
        self.pub_wheel_cmds = rospy.Publisher(
            "f/{veh_name}/wheels_driver_node/wheels_cmd",
            WheelsCmdStamped
        )
```

This initializes a `Publisher` that will publish data of type `WheelsCmdStamped` onto a topic called `/{veh_name}/wheels_driver_node/wheels_cmd`.



### Processing the incoming data and calculating the wheel commands

The main task we need to do is to build this function called `process_joy` that will process the incoming messages. Start by defining the function:

```python
    def process_joy(self,msg):
```

Notice that in addition to the normal `self` variable for a function that is defined in a class, there is also the `msg` variable. This is the variable that will contain the incoming data, in this case the joystick commands of type `Joy`. 

To make the code as clean as possible, let's start by defining and initializing the variable that we are going to end up publishing

```python
        cmd_to_publish = WheelsCmdStamped()
        cmd_to_publish.header = msg.header
        cmd_to_publish.vel_right = 0.0
        cmd_to_publish.vel_left = 0.0
```

The `header` part of the variable contains some meta information, such as a timestamp. It is good practice to copy this over from the incoming data so that we can calculate things like latency, but don't worry too much about this for now. As we saw from the definition of the [WheelsCmdStamped.msg](https://github.com/duckietown/dt-ros-commons/blob/daffy/packages/duckietown_msgs/msg/WheelsCmdStamped.msg) message, we need to set a `vel_right` and a `vel_left` (for each of the two wheels on the robot). We can start by initializing those to `0.0` so that in the case that none of the joystick buttons are active we will not move. 


Now we just need to check which of the buttons was pressed and then set the correct values (either + or -) to the corresponding wheels to make the robot go forward or backwards or turn left or right. Recall from the previous investigation that this information comes in on the `axes` data with the [Joy.msg](https://docs.ros.org/en/noetic/api/sensor_msgs/html/msg/Joy.html). Specifically, you will need to check the values of the `axes` data and set the values of the `cmd_to_publish.vel_right` and `cmd_to_publish.vel_left` in a way that is sensible. 



### Publishing the data

And, finally, the last thing we need to do is to publish the data. We can use the `Publisher` that we defined earlier:

```python
        self.pub_wheel_cmds.publish(cmd_to_publish)
```




### Testing if the Code Builds

In the **laptop** terminal (the one that you ran `dts code workbench` in - you can stop this is it is still running by pressing CTRL-C) you can test if your code doesn't have any syntax errors by running

    dts code build -R ROBOTNAME

You should see a bunch of output, but in there, but the important part, if all goes well, will be something that looks like this:

```bash
0.745 Starting >>> dt-joystick-demo                                                                                                                                  
0.745 Starting >>> duckietown                                                                                                                                        
0.745 Starting >>> duckietown_msgs                                                                                                                                   
0.746 Starting >>> duckietown_protocols                                                                                                                              
Finished <<< duckietown                          [ 0.2 seconds ]                                                                                                     
Finished <<< duckietown_protocols                [ 0.2 seconds ]                                                                                                     
Finished <<< duckietown_msgs                     [ 1.5 seconds ]                                                                                                     
Finished <<< dt-joystick-demo                    [ 2.3 seconds ]                                                                                                     
3.039 [build] Summary: All 4 packages succeeded!     
```

If you had a syntax error in your code, it will not succeed and it will report the error. 

## Testing your code

In the terminal on your laptop, you are now ready to test your code. Just as before run

    dts code workbench -R ROBOTNAME [-m]

Where the `-m` is optionally needed if you are running with a virtual robot in the Duckiematrix.

Open up the noVNC browser as before. Open up `rqt_image_view` and the joystick and try holding down buttons on the joystick. If all went well, you should see that you can control your Duckiebot with the joystick. If not, go back to the terminal where you ran `dts code workbench` to see if there are any errors reported by your code. If still not, or if the Duckiebot is not performing as you expected, you probably have an error in your code and you will need to do some more debugging. A good place to start could be to look at the data being published to the `/ROBOTNAME/wheels_driver_node/wheels_cmd` topic using `rostopic echo` or `rqtplot` as we have seen earlier. 



Congratulations! You have successfully written a ROS node to send control values from the joystick to make your Duckiebot move (either in simulation or with real hardware).

You may notice that there are some sub-optimalities about the implementation that we did. It is very simple, but can you think of things that could be improved? Feel free to try out some modifications!